In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, FuncFormatter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from scipy.stats import norm
import os

from model import *
from data import generate_gmm_data, generate_random_data
from train_utils import train, train_epoch

from plot import *
from utils import *



In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
relu = lambda X: np.maximum(X, 0)
d_relu = lambda X: np.maximum(np.sign(X), 0)

elu = lambda X, alpha=1.0: ( alpha * (np.exp(X) - 1) ) * (X <= 0) + (X * (X > 0))
d_elu = lambda X, alpha=1.0: ( alpha * np.exp(X) ) * (X <= 0) + (X > 0)

gelu = lambda X: X * norm.cdf(X)
d_gelu = lambda X: norm.cdf(X) + X * norm.pdf(X)

sigmoid = lambda X: 1 / (1 + np.exp(-X))
d_sigmoid = lambda X: sigmoid(X) * (1 - sigmoid(X))

silu = lambda X: X * sigmoid(X)
d_silu = lambda X: sigmoid(X) + X * d_sigmoid(X)

leaky_relu = lambda X, slope=0.01: slope * np.minimum(X, 0) + np.maximum(X, 0)
d_leaky_relu = lambda X, slope=0.01: slope * (X <= 0) + (X > 0).astype(float)

identity = lambda X: X
d_identity = lambda X: np.ones(X.shape)

abs_ = lambda X: np.abs(X)
d_abs_ = lambda X: -1 * (X <= 0) + (X > 0)

softplus = lambda X: np.log(1 + np.exp(X))
d_softplus = lambda X: sigmoid(X)

leaky_softplus = lambda X, slope=0.01: slope * X + (1 - slope) * (softplus(X) - np.log(2))
d_leaky_softplus = lambda X, slope=0.01: slope + (1 - slope) * (d_softplus(X))


smooth_abs = lambda X, beta = 10: (2 / beta) * softplus(X) - X - ( (2 / beta) * np.log(2) )
d_smooth_abs = lambda X, beta = 10: (2 / beta) * d_softplus(X) - 1

custom_activations_map = {'ReLU': [relu, d_relu],
               'Leaky-Softplus': [leaky_softplus, d_leaky_softplus], 
               'ELU': [elu, d_elu],
               'GELU': [gelu, d_gelu],
               'SiLU': [silu, d_silu],
               'Leaky-ReLU': [leaky_relu, d_leaky_relu],
               'Abs': [abs_, d_abs_],
               'SmoothAbs': [smooth_abs, d_smooth_abs],
               'Identity': [identity, d_identity]}

In [ ]:
reduction = 'sum'
loss_map = {
    'ce_loss': nn.CrossEntropyLoss(reduction=reduction),
    'mse_loss': nn.MSELoss(reduction=reduction)
}

optimizer_map = {'SGD': optim.SGD,
                'Adam': optim.Adam,
                'Muon': optim.Muon} #, 
                #'Muon': optim.Muon}

# Model Training

In [ ]:
def get_data(d: int, 
             K: int, 
             N_k: int, 
             nu: float = 1.0,
             normalize_: bool = False, 
             orthogonalize_: bool = False, 
             batch_size: int = None):
    
    
    N = K * N_k
    
    # d > N --> random Gaussian data
    if d > N:
        train_set, train_loader, grad_loader = generate_random_data(d, 
                                                       K, 
                                                       N_k, 
                                                       orthogonalize_ = orthogonalize_, 
                                                       normalize_ = normalize_, 
                                                       batch_size = batch_size)
        val_loader = train_loader 
    
    # N > d --> Gaussian mixture data
    elif N > d:
        train_set, train_loader, grad_loader = generate_gmm_data(d, 
                                                    K, 
                                                    N_k, 
                                                    nu=nu,
                                                    whiten_ = whiten_, 
                                                    normalize_ = normalize_, 
                                                    batch_size = batch_size)
        val_loader = train_loader
    
    
    X = train_set.samples.T
    Y = train_set.targets.T
    
    return train_loader, val_loader, grad_loader, X, Y

In [ ]:
def get_init_grads(model, dataloader, optimizer, criterion):
    
    results = train_epoch(model, 
                        dataloader, 
                        optimizer, 
                        criterion, 
                        device,
                        update=False, 
                        return_grads=True)
    
    return results['epoch_grads'] 
    

In [ ]:
def train_model(num_layers: int,
                data_dim: int, 
                hidden_dim: int, 
                num_classes: int, 
                act_fn_str: str, 
                train_loader, 
                grad_loader, 
                val_loader, 
                epochs, 
                lr, 
                freq,
                loss_fn, 
                optimizer_type='SGD', 
                freeze_last_layer=True, 
                diff_init_last_layer=False,
                residual=False, 
                batchnorm=False, 
                init_method='orthogonal',
                init_scale=1):
    
    
    # Train model
    model = MLP(num_layers, 
                data_dim, 
                hidden_dim, 
                num_classes, 
                act_fn_str, 
                freeze_last_layer=freeze_last_layer,
                diff_init_last_layer=diff_init_last_layer,
                last_layer_init_method='gaussian',
                residual=residual, 
                batchnorm=batchnorm, 
                init_method=init_method, 
                init_scale=init_scale)

    model = model.to(device)

    # Training
    criterion = loss_map[loss_fn]
    #optimizer = optimizer_map[optimizer_type](model.parameters(), lr=lr, momentum=0, weight_decay=0)
    #optimizer = optimizer_map[optimizer_type](model.parameters(), lr=lr, momentum=0.9, weight_decay=0)
    optimizer = optimizer_map[optimizer_type](model.parameters(), lr=lr, weight_decay=0)
    scheduler = None
    
    init_grads = get_init_grads(model, grad_loader, optimizer, criterion)
    results = train(model, 
                    criterion, 
                    optimizer, 
                    train_loader, 
                    val_loader, 
                    epochs, 
                    device,
                    scheduler=scheduler, 
                    return_weights=True,
                    return_grads=False,
                    return_outs=False)


    all_weights = results['all_weights']
    #grads = results['all_grads']
    #outs = results['all_outs']
    train_losses = results['train_losses']
    
    return init_grads, all_weights, train_losses #, outs

# Weight decompositions during training

In [ ]:
def fro_inner_product(X: np.ndarray, Y: np.ndarray):
    assert X.shape == Y.shape
    
    X_norm = np.linalg.norm(X, ord='fro')
    Y_norm = np.linalg.norm(Y, ord='fro')
    
    return np.trace(X.T @ Y) / (X_norm * Y_norm)

def normalized_dist(X: np.ndarray, Y: np.ndarray, norm_factor = 1.0):
    assert X.shape == Y.shape
    
    return np.linalg.norm(X - Y)**2 / norm_factor #np.sqrt( np.linalg.norm(X - Y)**2 / norm_factor )

def normalized_norms(X: np.ndarray):
    return normalized_dist(X, np.zeros(X.shape))
    

In [ ]:
def weight_decomp_norms_dists(all_weights, init_grads, K, init_scale):
    
    L = len(all_weights[0])
    epochs = len(all_weights) - 1
    
    # Get "small update subspace"
    W1_0 = all_weights[0][0]
    G1_0 = init_grads[0]
    
    U_null, V_null = small_update_subspace(W1_0, G1_0, K, init_scale)
    p = U_null.shape[-1]

    # Get "large update subspace"
    U_update = np.linalg.svd(U_null, full_matrices=True)[0][:, p:]
    V_update = np.linalg.svd(V_null, full_matrices=True)[0][:, p:]
    
    
    # Concatenate two subspaces to create orthogonal matrices
    U = np.concatenate((U_update, U_null), axis=1)
    V = np.concatenate((V_update, V_null), axis=1)

    
    # Get initial weight factors
    all_Wl_0_tilde = []
    
    U_l, V_l = U.copy(), V.copy()
    for l in range(L - 1):
        # Partition in update and small-update
        Vl_update = V_l[:, :p]
        Ul_update = U_l[:, :p]
        
        Vl_null = V_l[:, p:]
        Ul_null = U_l[:, p:]
        
        # Get weights and resulting decomposition
        W_l_0 = all_weights[0][l]
        
        Wl_0_tilde = []
        
        Wl_0_tilde.append(Ul_update.T @ W_l_0 @ Vl_update)
        Wl_0_tilde.append(Ul_update.T @ W_l_0 @ Vl_null)
        Wl_0_tilde.append(Ul_null.T @ W_l_0 @ Vl_update)
        Wl_0_tilde.append(Ul_null.T @ W_l_0 @ Vl_null)
        
        all_Wl_0_tilde.append(Wl_0_tilde)

        # Update U and V factors 
        V_l = U_l 
        U_l = all_weights[0][l + 1] @ V_l / init_scale
    
    
    
    # Calculate weight decomposition norms and distance from initialization
    # for all weights until last-layer classifier 
    dists = np.zeros((epochs - 1, L - 1, 4))
    for t in range(1, epochs):
        U_l, V_l = U.copy(), V.copy()
        for l in range(L - 1):
            
            # Partition into update and small-update
            Vl_update = V_l[:, :p]
            Ul_update = U_l[:, :p]

            Vl_null = V_l[:, p:]
            Ul_null = U_l[:, p:]
        
            # Get weights and resulting decomposition
            W_l_t = all_weights[t][l]
        

            Wl_11_tilde = Ul_update.T @ W_l_t @ Vl_update
            Wl_12_tilde = Ul_update.T @ W_l_t @ Vl_null
            Wl_13_tilde = Ul_null.T @ W_l_t @ Vl_update
            Wl_14_tilde = Ul_null.T @ W_l_t @ Vl_null
            
            
            # Calculate relative squared distance of each component
            Wl_0_tilde = all_Wl_0_tilde[l]
            
            norm_factor = np.linalg.norm(W_l_t - all_weights[0][l], ord='fro')**2
            
            dists[t - 1, l, 0] = normalized_dist(Wl_11_tilde, Wl_0_tilde[0], norm_factor)
            
            dists[t - 1, l, 1] = normalized_dist(Wl_12_tilde, Wl_0_tilde[1], norm_factor)
            
            dists[t - 1, l, 2] = normalized_dist(Wl_13_tilde, Wl_0_tilde[2], norm_factor)
            
            dists[t - 1, l, 3] = normalized_dist(Wl_14_tilde, Wl_0_tilde[3], norm_factor)
            
            
            # Update U and V factors for next layer 
            V_l = U_l 
            U_l = all_weights[0][l + 1] @ V_l / init_scale
        
        
    return dists

In [ ]:
# Data setup
d = 64 # Ambient dim
m = 72
K = 4 # Number of classes
N_k = 25
N = N_k * K
nu = np.sqrt(3)

# Model setup
L = 3
act_fn_strs = ['GELU', 'SiLU', 'ELU'] #, 'LeakyReLU', 'ReLU']
#act, d_act = custom_activations_map[act_fn_str]
residual = False
batchnorm = False
freeze_last_layer = False
diff_init_last_layer = True
init_method = 'orthogonal'
init_scale = 0.1

whiten_ = False
normalize_ = False
orthogonalize_ = False
batch_size = 32

# Training setup
epochs = 250

# GD w/ MSE loss
#lrs = [5e-3, 5e-3, 1e-3, 1e-3, 1e-3] #[5e-4, 5e-4, 1e-4, 5e-4, 5e-4] 

# SGD w/ CE loss
lrs = [5e-3] * len(act_fn_strs) #[5e-4, 5e-4, 1e-4, 5e-4, 5e-4] #[5e-3] * len(act_fn_strs) # [1e-3, 1e-3, 5e-4, 1e-3, 1e-3] # [5e-4, 5e-4, 1e-4, 5e-4, 5e-4]
freq = 1

# CHANGE THIS 
# torch.optim.Adam does not have a momentum argument, but Muon does. 
# For Adam, keep the "betas" argument at the default values, which should be (0.9, 0.999).
# Set the Muon "momentum" to 0.9.
# For all optimizers, set the weight decay to 0 (should be the default value in Pytorch anyways.)
optimizer_type = 'SGD' # Change to 'Adam' or 'Muon'. Set momentum to 0.9. 


loss_fn = 'ce_loss'
#loss_fn = 'mse_loss'

savefig_dir = './figs/beyond_theory/' + loss_fn + '/'
savefig_dir += 'full_GD' if (optimizer_type == 'SGD') and (batch_size == N or not batch_size) else optimizer_type + '_batch_size_' + str(batch_size)
if not os.path.exists(savefig_dir):
    os.makedirs(savefig_dir)
    
save = False

run = True


In [ ]:
if run: 
    trials = 1

    weight_decomp_dists = np.zeros((trials, len(act_fn_strs), epochs - 1, L - 1, 4))
    train_losses = np.zeros((trials, len(act_fn_strs), epochs))
    
    nonlin_comps = np.zeros((trials, len(act_fn_strs), epochs))

    for (k, act_fn_str) in enumerate(act_fn_strs):
        lr = lrs[k]
        print("===== ACTIVATION FUNCTION: ", act_fn_str, " =====")
        for i in range(trials):
            # Create data
            train_loader, val_loader, grad_loader, X, Y = get_data(d, K, N_k, nu, whiten_, normalize_, batch_size=batch_size)

            print("=== TRIAL " + str(i) + " ===")
            # Train model
            init_grads, all_weights, train_losses_trial = train_model(L, d, m, K, act_fn_str, 
                                                                     train_loader, 
                                                                     grad_loader, 
                                                                     val_loader, 
                                                                     epochs, 
                                                                     lr, 
                                                                     freq, 
                                                                     loss_fn, 
                                                                     optimizer_type=optimizer_type,
                                                                     freeze_last_layer=freeze_last_layer,
                                                                     diff_init_last_layer=diff_init_last_layer,
                                                                     residual=residual,
                                                                     batchnorm=batchnorm,
                                                                     init_method=init_method,
                                                                     init_scale=init_scale)



            # Plot train loss
            plot_train_losses(train_losses_trial, act_fn_str, d, K, None, savefig=False)

            train_losses[i, k, :] = train_losses_trial


            # Calculate norms of decompositions
            trial_decomp_dists = weight_decomp_norms_dists(all_weights, init_grads, K, init_scale)

            weight_decomp_dists[i, k, ...] = trial_decomp_dists
            
            #
            for t in range(epochs):
                weights_t = all_weights[t]
                
                lin_out = X
                nonlin_out = X
                for l in range(L):
                    W_l_t = weights_t[l]
                    lin_out = W_l_t @ lin_out
                    
                    nonlin_out = W_l_t @ nonlin_out
                    if l < L - 1: 
                        nonlin_out = activations_map[act_fn_str](torch.Tensor(nonlin_out)).numpy()
                
                if k < 2: 
                    lin_out *= 0.5
            
                
                diff = nonlin_out - lin_out
                print(np.linalg.norm(diff)**2 / np.linalg.norm(nonlin_out)**2)
            
            print("\n")


    # Save results
    save_fname = os.path.join(savefig_dir, 'all_acts_losses_decomp_dists')
    save_fname += '_unwhitened_data' if not whiten_ else ''
    save_fname += '_large_d.npz' if d > N else '_large_N.npz'

    if save:
        np.savez(save_fname, 
                 train_losses=train_losses, 
                 weight_decomp_dists=weight_decomp_dists)
        
else:
    load_fname = os.path.join(savefig_dir, 'all_acts_losses_decomp_dists')
    load_fname += '_unwhitened_data' if not whiten_ else ''
    load_fname += '_large_d.npz' if d > N else '_large_N.npz'
    
    results = np.load(load_fname)
    
    weight_decomp_dists = results['weight_decomp_dists']

In [ ]:
loss_str_map = {
    'mse_loss': 'Squared Error Loss',
    'ce_loss': 'Cross Entropy Loss'
}

In [ ]:
def log_errorbar(y, std):
    y_low = y * np.exp(-std / y)
    y_high = y * np.exp(std / y)
    return np.vstack((y - y_low, y_high - y))

def at_most_2_decimals(x, pos):
    return f"{x:.2f}" #.rstrip('0').rstrip('.')

def two_to_three_decimals(x, pos):
    s = f"{x:.3f}"          # round to 3 decimals
    integer, _, frac = s.partition('.')

    # ensure at least 2 decimals
    frac = frac.rstrip('0')
    if len(frac) < 2:
        frac = frac.ljust(2, '0')

    return integer + '.' + frac

formatter = FuncFormatter(at_most_2_decimals)

In [ ]:
# Mean and stdev of weight decomposition norms across trials
mean_weight_decomp_dists = np.mean(weight_decomp_dists, axis=0)
std_weight_decomp_dists = np.std(weight_decomp_dists, axis=0) 


# Plotting
x_range1 = np.arange(weight_decomp_dists.shape[2]) + 1
x_range2 = np.arange(0, epochs) 


for (k, act_fn_str) in enumerate(act_fn_strs):    
    # Create two figures
    fig1, ax1 = plt.subplots(nrows = 1, ncols = L - 1, figsize=(int(7*(L - 1)), 7), sharey=True)
    handles1, labels1 = [], [] 

    
    # Plot each layer
    for l in range(L - 1):
        for j in range(4):
            # Plot normalized distance from initialization
            plt_label = r'$\widetilde{W}_{l,' + str(j + 1) + r'}(t)$'
            ax1[l].plot(x_range1, 
                       mean_weight_decomp_dists[k, :, l, j], 
                       linewidth=8,
                       label=plt_label)
            
            ax1[l].fill_between(x_range1, 
                             mean_weight_decomp_dists[k, :, l, j] * np.exp(-std_weight_decomp_dists[k, :, l, j]), 
                             mean_weight_decomp_dists[k, :, l, j] * np.exp(std_weight_decomp_dists[k, :, l, j]),
                             linewidth=8,
                             alpha=0.5)
            
#             ax1[l].fill_between(x_range1, 
#                              mean_weight_decomp_dists[k, :, l, j] -std_weight_decomp_dists[k, :, l, j], 
#                              mean_weight_decomp_dists[k, :, l, j] + std_weight_decomp_dists[k, :, l, j],
#                              linewidth=8,
#                              alpha=0.5)
            

        # Figure 1 visuals
        ax1[l].grid('on')
        ax1[l].tick_params(axis='x', labelsize=32)
        ax1[l].tick_params(axis='y', labelsize=32)
        ax1[l].yaxis.set_major_formatter(formatter)
        ax1[l].set_title('Layer ' + str(l + 1), fontsize=36)
        #ax1[l].set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
        #ax1[l].set_ylim([-0.1, 1.1])
        ax1[l].set_yscale('log')
        ax1[l].set_ylim([3e-4, 3e0])


        for handle1, label1 in zip(*ax1[l].get_legend_handles_labels()):
            if label1 not in labels1:
                handles1.append(handle1)
                labels1.append(label1)
                
    
    
    # Show figure 1    
    if optimizer_type == 'SGD' and (not batch_size or batch_size == N):
        exp_info = act_fn_str + ' (' + loss_str_map[loss_fn] + ', Full Batch GD,'
    else:
        exp_info = act_fn_str + ' (' + loss_str_map[loss_fn] + ', ' + optimizer_type + ',' #+ ' with Momentum,'
    
    exp_info += ' Whitened Data)' if whiten_ else ' Unwhitened Data)'
    
    #exp_info += r'$d > N$)' if d > N else r'$N > d$)'
        
    fig1.suptitle(exp_info, fontsize=40, x=0.55, y=1.08)
    fig1.supxlabel('Train Epoch', fontsize=40)
    fig1.supylabel('Norm. Squared\nDist. from Init.', 
                  fontsize=40, ha='center',
                 x = 0.03)

    fig1.legend(
        handles1,
        labels1,
        loc='upper center',
        bbox_to_anchor=(0.55, 1.05), 
        ncol=4,
        fontsize=36,
        frameon=False
    )


    fig1.tight_layout()    

    save_fname = act_fn_str + '_dist_from_init'
    save_fname += '_unwhitened_data.pdf' if not whiten_ else '.pdf'
    #save_fname = save_fname + '_large_d.pdf' if d > N else save_fname + '_large_N.pdf'
    if save:
        plt.savefig(os.path.join(savefig_dir, save_fname), bbox_inches='tight')
    plt.show()
    plt.close()